# ECON326 - Final Project

## Import Statements

In [1]:
install.packages("modelsummary")
install.packages("sandwich") 
install.packages("kableExtra")
install.packages("estimatr")
library(tidyverse)
library(haven)
library(dplyr)
library(scales)
library(stargazer)
library(car)
library(lmtest) 
library(sandwich) 
library(modelsummary)
library(knitr)
library(kableExtra)
library(estimatr)

census_data <- read_csv("data/data_donnees_2021_ind_v2.csv")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:


In [2]:
options(modelsummary_factory_default = "latex_tabular")

In [3]:
head(census_data)

PPSORT,ABOID,AGEGRP,AGEIMM,ATTSCH,BFNMEMB,BedRm,CFInc,CFInc_AT,CFSTAT,⋯,WT7,WT8,WT9,WT10,WT11,WT12,WT13,WT14,WT15,WT16
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,6,13,7,1,0,4,30,27,2,⋯,22.90113,22.90113,22.90113,249.27876,22.90113,22.90113,22.90113,22.90113,22.90113,22.90113
2,6,11,5,1,0,3,18,18,2,⋯,22.89379,22.89379,22.89379,22.89379,22.89379,22.89379,22.89379,22.89379,22.89379,22.89379
3,1,13,99,1,0,0,7,7,6,⋯,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134
4,6,16,99,1,0,4,15,15,2,⋯,22.87713,22.87713,22.87713,22.87713,22.87713,22.87713,22.87713,22.87713,22.87713,22.87713
5,6,18,99,1,0,3,13,13,3,⋯,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134,22.90134
6,2,16,99,1,0,4,1,1,7,⋯,22.89284,22.89284,22.89284,22.89284,22.89284,22.89284,22.89284,249.18847,22.89284,22.89284


## Data Filtering & Cleaning

In [4]:
# includes only variables needed for our regression
census_data <- census_data |>
    select(TotInc, POB, HDGREE, CIP2021, LOCSTUD, NOC21, KOL, AGEIMM, Gender, MarStH, HHSIZE)
head(census_data)

TotInc,POB,HDGREE,CIP2021,LOCSTUD,NOC21,KOL,AGEIMM,Gender,MarStH,HHSIZE
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
76000,88,7,7,17,88,1,7,2,2,6
32000,28,7,8,16,14,1,5,1,2,4
17000,1,2,13,99,99,2,99,1,1,1
22000,1,5,10,9,99,1,99,2,2,5
22000,1,2,13,99,99,2,99,2,5,2
1000,1,2,13,99,99,1,99,1,6,3


In [5]:
# removing citizens with unavailable immigrant data
census_data <- census_data |>
    filter(AGEIMM < 88 & TotInc > 0 & TotInc < 1e6 & HDGREE < 20)
head(census_data)

TotInc,POB,HDGREE,CIP2021,LOCSTUD,NOC21,KOL,AGEIMM,Gender,MarStH,HHSIZE
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
76000,88,7,7,17,88,1,7,2,2,6
32000,28,7,8,16,14,1,5,1,2,4
67000,11,12,4,14,99,1,7,1,2,2
45000,12,2,13,99,1,1,6,2,2,4
64000,6,2,13,99,6,2,8,2,2,3
9000,21,9,5,15,2,1,9,1,2,3


In [6]:
# checking for extreme values of TotInc
summary(census_data$TotInc)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
      1   22000   38000   48962   62000  452570 

In [7]:
# Converts integers into categories as strings (words), set up as factor variables
census_data <- census_data |>
    mutate(Gender = factor(Gender, levels = c(1,2), labels = c("Female", "Male")),
          POB = factor(POB, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,88),
                       labels = c("Canada", "USA", "Central America", "Jamaica", "Caribbean", "South America", "UK", "Germany", "France", "Northern + Western Europe", "Poland", "Eastern Europe", 
                                  "Italy", "Portugal", "Southern Europe", "Eastern Africa", "Northern Africa", "South + West Africa", "Iran", "Middle East + West Central Asia", "China", "Hong Kong", 
                                  "South Korea", "East Asia", "Philippines", "Vietnam", "Southeast Asia", "India", "Pakistan", "Sri Lanka", "Other South Asia", "Oceania + others", "N/A")),
          MarStH = factor(MarStH, levels = c(1,2,3,4,5,6), labels = c("Never married", "Married", "Living common law", "Separated", "Divorced", "Widowed")),
          CIP2021 = relevel(factor(CIP2021, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,88,99),
                          labels = c("Education", "Visual + Performing Arts", "Humanities", "Social Sciences + Law", "Business + Public Admin", "Physical + Life Sciences", "Math + Technology", 
                                     "Architecture + Engineering", "Agriculture", "Health", "Services", "Other", "No certificate, diploma, or degree", "N/A", "N/A")),
                            ref = "No certificate, diploma, or degree"),
          LOCSTUD = factor(LOCSTUD, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,99),
                          labels = c("Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", 
                                     "US", "Other Americas", "Europe", "East Asia", "South + Southeast Asia", "Other", "N/A")),
          AGEIMM = relevel(factor(AGEIMM, levels= c(1,2,3,4,5,6,7,8,9,10,11,12,13,88,99),
                         labels = c("0 to 4 yo", "5 to 9 yo", "10 to 14 yo", "15 to 19 yo", "20 to 24 yo", "25 to 29 yo", "30 to 34 yo", 
                                    "35 to 39 yo", "40 to 44 yo", "45 to 49 yo", "50 to 54 yo", "55 to 59 yo", "60+ yo", "N/A", "N/A")),
                          ref = "N/A"),
          HDGREE = relevel(factor(HDGREE, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,88,99),
                                 labels = c("No certificate, diploma, or degree",
                                           "High school diploma",
                                           "Non-apprenticeship Trades",
                                           "Apprenticeship Certificate",
                                           "3 months to 1 year in postsecondary",
                                           "1 to 2 years in postsecondary",
                                           "More than 2 years in postsecondary",
                                           "Diploma below bachelor's",
                                           "Bachelor's Degree",
                                            "University diplom above Bachelor's",
                                           "Degree in medicine or related",
                                           "Master's Degree",
                                           "Earned Doctorate",
                                           "N/A",
                                           "N/A")),
                                  ref = "No certificate, diploma, or degree")) |>
    mutate(
        noc21 = case_when(
            NOC21 == 1 ~ "Management + Business",
            NOC21 == 2 ~ "Management + Business",
            NOC21 == 3 ~ "Management + Business",
            NOC21 == 4 ~ "Management + Business",
            NOC21 == 5 ~ "Management + Business",
            NOC21 == 6 ~ "Management + Business",
            NOC21 == 7 ~ "Natural + Applied Sci",
            NOC21 == 8 ~ "Natural + Applied Sci",
            NOC21 == 9 ~ "Health",
            NOC21 == 10 ~ "Health",
            NOC21 == 11 ~ "Health",
            NOC21 == 12 ~ "Educ + Law + Social + Govt Services",
            NOC21 == 13 ~ "Educ + Law + Social + Govt Services",
            NOC21 == 14 ~ "Educ + Law + Social + Govt Services",
            NOC21 == 15 ~ "Arts + Culture + Recreation",
            NOC21 == 16 ~ "Arts + Culture + Recreation",
            NOC21 == 17 ~ "Sales + Services",
            NOC21 == 18 ~ "Sales + Services",
            NOC21 == 19 ~ "Sales + Services",
            NOC21 == 20 ~ "Sales + Services",
            NOC21 == 21 ~ "Trades + Transportation",
            NOC21 == 22 ~ "Trades + Transportation",
            NOC21 == 23 ~ "Trades + Transportation",
            NOC21 == 24 ~ "Trades + Transportation",
            NOC21 == 25 ~ "Natural Resources + Production",
            NOC21 == 26 ~ "Natural Resources + Production",
            NOC21 == 88 ~ "N/A",
            NOC21 == 99 ~ "N/A",
        )) |>
    mutate(noc21 = relevel(as_factor(noc21), ref = "Sales + Services"))
head(census_data)

TotInc,POB,HDGREE,CIP2021,LOCSTUD,NOC21,KOL,AGEIMM,Gender,MarStH,HHSIZE,noc21
<dbl>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<dbl>,<fct>
76000,N/A,More than 2 years in postsecondary,Math + Technology,Other,88,1,30 to 34 yo,Male,Married,6,N/A
32000,India,More than 2 years in postsecondary,Architecture + Engineering,South + Southeast Asia,14,1,20 to 24 yo,Female,Married,4,Educ + Law + Social + Govt Services
67000,Poland,Master's Degree,Social Sciences + Law,Europe,99,1,30 to 34 yo,Female,Married,2,N/A
45000,Eastern Europe,High school diploma,"No certificate, diploma, or degree",N/A,1,1,25 to 29 yo,Male,Married,4,Management + Business
64000,South America,High school diploma,"No certificate, diploma, or degree",N/A,6,2,35 to 39 yo,Male,Married,3,Management + Business
9000,China,Bachelor's Degree,Business + Public Admin,East Asia,2,1,40 to 44 yo,Female,Married,3,Management + Business


In [8]:
# checking for multicollinearity through VIF test
vif_model_simple <- lm(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE, data = census_data)
vif(vif_model_simple, type = "predictor")

GVIFs computed for predictors



,GVIF,Df,GVIF^(1/(2*Df)),Interacts With,Other Predictors
,<dbl>,<dbl>,<dbl>,<chr>,<chr>
POB,6.091227e+01,31,1.068527,--,"HDGREE, CIP2021, LOCSTUD, noc21, KOL, AGEIMM, Gender, MarStH, HHSIZE"
HDGREE,-2.026721e+17,12,NaN,--,"POB, CIP2021, LOCSTUD, noc21, KOL, AGEIMM, Gender, MarStH, HHSIZE"
CIP2021,1.339892e+17,12,5.171636,--,"POB, HDGREE, LOCSTUD, noc21, KOL, AGEIMM, Gender, MarStH, HHSIZE"
LOCSTUD,1.962414e+17,7,17.186971,--,"POB, HDGREE, CIP2021, noc21, KOL, AGEIMM, Gender, MarStH, HHSIZE"
noc21,5.575239e+00,8,1.113375,--,"POB, HDGREE, CIP2021, LOCSTUD, KOL, AGEIMM, Gender, MarStH, HHSIZE"
KOL,1.357903e+00,1,1.165291,--,"POB, HDGREE, CIP2021, LOCSTUD, noc21, AGEIMM, Gender, MarStH, HHSIZE"
AGEIMM,9.766920e+00,12,1.099613,--,"POB, HDGREE, CIP2021, LOCSTUD, noc21, KOL, Gender, MarStH, HHSIZE"
Gender,1.374905e+00,1,1.172563,--,"POB, HDGREE, CIP2021, LOCSTUD, noc21, KOL, AGEIMM, MarStH, HHSIZE"
MarStH,2.206743e+00,5,1.082369,--,"POB, HDGREE, CIP2021, LOCSTUD, noc21, KOL, AGEIMM, Gender, HHSIZE"


## Summary Statistics

In [9]:
# Creates summary for each categorical variable, help from ChatGPT,
# these summaries are included in the appendix 

# Create output folder if it doesn't exist
if (!dir.exists("tables")) dir.create("tables")

# List of categorical variables
cat_vars <- c("POB", "HDGREE", "CIP2021", "LOCSTUD", "noc21", "AGEIMM", "Gender", "MarStH", "KOL", "HHSIZE")

# Loop over each variable to create and export frequency tables
for (var in cat_vars) {
  freq_table <- census_data %>%
    count(!!sym(var)) %>%
    mutate(Percent = round(100 * n / sum(n), 2))

  # Rename columns for display
  colnames(freq_table) <- c("Category", "Count", "Percent")

  # Create LaTeX table and export it
  kbl(freq_table, format = "latex", booktabs = TRUE,
      caption = paste("Frequency Table for", var),
      label = paste0("tab:", var)) %>%
    kable_styling(latex_options = c("hold_position")) %>%
    save_kable(file = paste0("tables/", var, "_summary.tex"))
}

In [10]:
# for continuous variables, just re-run again if run into warning message
continuous_variables_table <- datasummary(TotInc ~ Mean + SD + Min + Max + N,
                             data = census_data,
                             output = "latex_tabular",
                             title = "Numerical Variable Summary")

writeLines(as.character(continuous_variables_table), "tables/numeric_summary.tex")

Warning message:
“To compile a LaTeX document with this table, the following commands must be placed in the document preamble:

\usepackage{tabularray}
\usepackage{float}
\usepackage{graphicx}
\usepackage{codehigh}
\usepackage[normalem]{ulem}
\UseTblrLibrary{booktabs}
\UseTblrLibrary{siunitx}
\newcommand{\tinytableTabularrayUnderline}[1]{\underline{#1}}
\newcommand{\tinytableTabularrayStrikeout}[1]{\sout{#1}}
\NewTableCommand{\tinytableDefineColor}[3]{\definecolor{#1}{#2}{#3}}

To disable `siunitx` and prevent `modelsummary` from wrapping numeric entries in `\num{}`, call:

options("modelsummary_format_numeric_latex" = "plain")
 This warning appears once per session.”


## Model Explanation

In [11]:
# only include immigrants in our model
immigrants_only <- subset(census_data, POB != "Canada")

# original model
RegImmigrants <- lm(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(RegImmigrants)
RegImmigrants_robust <- lm_robust(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(RegImmigrants_robust)


Call:
lm(formula = TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + 
    KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + 
    POB:LOCSTUD, data = immigrants_only)

Residuals:
    Min      1Q  Median      3Q     Max 
-135456  -13555       0   10512  252372 

Coefficients: (325 not defined because of singularities)
                                                                              Estimate
(Intercept)                                                                    33006.7
POBCentral America                                                              5135.3
POBJamaica                                                                      7109.9
POBCaribbean                                                                    3165.3
POBSouth America                                                                6740.4
POBUK                                                                           1808.6
POBGermany                                       

325 coefficients  not defined because the design matrix is rank deficient





Call:
lm_robust(formula = TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + 
    noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + 
    AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)

Standard error type:  HC2 

Coefficients: (325 not defined because the design matrix is rank deficient)
                                                                              Estimate
(Intercept)                                                                   -16362.1
POBCentral America                                                             32155.0
POBJamaica                                                                     55131.5
POBCaribbean                                                                   31571.6
POBSouth America                                                               70298.7
POBUK                                                                          79542.4
POBGermany                                                                     57788.6
PO

In [12]:
# model #2, with NOC21 instead of simplified noc21 variable
RegImmigrantsNOC <- lm(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(RegImmigrantsNOC)

RegImmigrantsNOC_robust <- lm_robust(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")
summary(RegImmigrants_robust)


Call:
lm(formula = TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + 
    KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + 
    POB:LOCSTUD, data = immigrants_only)

Residuals:
    Min      1Q  Median      3Q     Max 
-137969  -14384       0   10218  252219 

Coefficients: (325 not defined because of singularities)
                                                                               Estimate
(Intercept)                                                                    44965.83
POBCentral America                                                              6587.55
POBJamaica                                                                      2548.94
POBCaribbean                                                                    5713.59
POBSouth America                                                                7959.33
POBUK                                                                           3503.11
POBGermany                                

325 coefficients  not defined because the design matrix is rank deficient





Call:
lm_robust(formula = TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + 
    noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + 
    AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)

Standard error type:  HC2 

Coefficients: (325 not defined because the design matrix is rank deficient)
                                                                              Estimate
(Intercept)                                                                   -16362.1
POBCentral America                                                             32155.0
POBJamaica                                                                     55131.5
POBCaribbean                                                                   31571.6
POBSouth America                                                               70298.7
POBUK                                                                          79542.4
POBGermany                                                                     57788.6
PO

In [13]:
# transforming our regression into a log-linear model to eliminate total income's right-skewed distribution

immigrants_only$log_TotInc <- log(census_data$TotInc)

# model #3, the model we ultimately used for our analysis in the discussion and table of results
linlogRegImmigrants <- lm(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
    
summary(linlogRegImmigrants)


Call:
lm(formula = log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + 
    noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + 
    AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)

Residuals:
    Min      1Q  Median      3Q     Max 
-9.4007 -0.2722  0.0000  0.4714  5.4738 

Coefficients: (325 not defined because of singularities)
                                                                              Estimate
(Intercept)                                                                   9.244549
POBCentral America                                                            0.171488
POBJamaica                                                                   -0.121401
POBCaribbean                                                                 -0.078648
POBSouth America                                                             -0.041225
POBUK                                                                         0.175490
POBGermany                                   

In [14]:
# a robust test to eliminate heteroskedasticity
linlogRegImmigrants_robust <- lm_robust(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")
summary(linlogRegImmigrants_robust)

325 coefficients  not defined because the design matrix is rank deficient





Call:
lm_robust(formula = log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + 
    noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + 
    AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")

Standard error type:  HC1 

Coefficients: (325 not defined because the design matrix is rank deficient)
                                                                              Estimate
(Intercept)                                                                  12.047224
POBCentral America                                                            0.287588
POBJamaica                                                                    2.985087
POBCaribbean                                                                  2.447448
POBSouth America                                                              1.358322
POBUK                                                                         4.590532
POBGermany                                                          

In [15]:
# model 4, used NOC21 variable instead of simplified noc21.
linlogRegImmigrantsNOC <- lm(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(linlogRegImmigrantsNOC)

linlogRegImmigrantsNOC_robust <- lm_robust(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")
summary(linlogRegImmigrantsNOC_robust)


Call:
lm(formula = log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + 
    NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + 
    AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)

Residuals:
    Min      1Q  Median      3Q     Max 
-9.4062 -0.2670  0.0000  0.4682  5.5011 

Coefficients: (325 not defined because of singularities)
                                                                               Estimate
(Intercept)                                                                   9.5637366
POBCentral America                                                            0.2150507
POBJamaica                                                                   -0.1816399
POBCaribbean                                                                 -0.0431175
POBSouth America                                                              0.0296967
POBUK                                                                         0.2171778
POBGermany                            

325 coefficients  not defined because the design matrix is rank deficient





Call:
lm_robust(formula = log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + 
    NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + 
    AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")

Standard error type:  HC1 

Coefficients: (325 not defined because the design matrix is rank deficient)
                                                                               Estimate
(Intercept)                                                                          NA
POBCentral America                                                           -0.1707185
POBJamaica                                                                   -3.3380540
POBCaribbean                                                                  2.6846350
POBSouth America                                                             -0.5735713
POBUK                                                                        -2.1009269
POBGermany                                                   

## Table of Results

In [16]:
# we use model summary as it is more flexible and easier to include robust regressions 
# to find significant results.

# model 1:
modelsummary(
    RegImmigrants_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)

\begin{tabular}{ll}
\hline
& (1) \\ \hline
(Intercept) & \num{-16362.100} \\
POBCentral America & \num{32154.989} \\
POBJamaica & \num{55131.527} \\
POBCaribbean & \num{31571.558} \\
POBSouth America & \num{70298.698} \\
POBUK & \num{79542.437} \\
POBGermany & \num{57788.636} \\
POBFrance & \num{55965.417} \\
POBNorthern + Western Europe & \num{70599.869} \\
POBPoland & \num{-20293.679} \\
POBEastern Europe & \num{82601.277} \\
POBItaly & \num{-28054.259} \\
POBPortugal & \num{86297.206} \\
POBSouthern Europe & \num{-16137.379} \\
POBEastern Africa & \num{-24522.391} \\
POBNorthern Africa & \num{-118199.688} \\
POBSouth + West Africa & \num{68972.963} \\
POBIran & \num{35792.300} \\
POBMiddle East + West Central Asia & \num{-27306.333} \\
POBChina & \num{81749.759} \\
POBHong Kong & \num{12242.727} \\
POBSouth Korea & \num{49032.297} \\
POBEast Asia & \num{-39661.113} \\
POBPhilippines & \num{-5176.982} \\
POBVietnam & \num{-71233.702} \\
POBSoutheast Asia & \num{-117155.198} \\
POBInd

In [17]:
# model 2:
modelsummary(
    RegImmigrantsNOC_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)

\begin{tabular}{ll}
\hline
& (1) \\ \hline
POBCentral America & \num{-22180.752} \\
& (\num{31671.586}) \\
POBJamaica & \num{-113678.905}*** \\
& (\num{33125.413}) \\
POBCaribbean & \num{57991.431} \\
& (\num{53452.751}) \\
POBSouth America & \num{-28216.691} \\
& (\num{30417.173}) \\
POBUK & \num{-9997.006} \\
& (\num{45258.322}) \\
POBGermany & \num{97072.507} \\
& (\num{96609.923}) \\
POBFrance & \num{-131509.274} \\
& (\num{147597.038}) \\
POBNorthern + Western Europe & \num{-208494.897} \\
& (\num{144414.282}) \\
POBPoland & \num{-216542.898} \\
& (\num{144238.486}) \\
POBEastern Europe & \num{-97159.492} \\
& (\num{60748.983}) \\
POBItaly & \num{97447.950} \\
& (\num{96690.344}) \\
POBPortugal & \num{70823.305} \\
& (\num{50092.861}) \\
POBSouthern Europe & \num{-294085.162}* \\
& (\num{148550.173}) \\
POBEastern Africa & \num{105929.651}* \\
& (\num{50166.341}) \\
POBNorthern Africa & \num{-127556.545}* \\
& (\num{58740.415}) \\
POBSouth + West Africa & \num{82964.812} \\
& (\nu

In [18]:
# shows summary including significant results and their levels of significance

# model 3, the model we used:
modelsummary(
    linlogRegImmigrants_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)

\begin{tabular}{ll}
\hline
& (1) \\ \hline
(Intercept) & \num{12.047}*** \\
& (\num{1.550}) \\
POBCentral America & \num{0.288} \\
& (\num{0.950}) \\
POBJamaica & \num{2.985}** \\
& (\num{1.052}) \\
POBCaribbean & \num{2.447}* \\
& (\num{0.960}) \\
POBSouth America & \num{1.358}* \\
& (\num{0.606}) \\
POBUK & \num{4.591}*** \\
& (\num{1.200}) \\
POBGermany & \num{0.726} \\
& (\num{0.500}) \\
POBFrance & \num{3.151}** \\
& (\num{1.110}) \\
POBNorthern + Western Europe & \num{-0.880} \\
& (\num{1.568}) \\
POBPoland & \num{-1.977} \\
& (\num{1.418}) \\
POBEastern Europe & \num{4.033}** \\
& (\num{1.545}) \\
POBItaly & \num{0.389} \\
& (\num{0.529}) \\
POBPortugal & \num{3.496}*** \\
& (\num{0.931}) \\
POBSouthern Europe & \num{-1.832} \\
& (\num{1.568}) \\
POBEastern Africa & \num{-1.456} \\
& (\num{1.371}) \\
POBNorthern Africa & \num{-3.135}* \\
& (\num{1.458}) \\
POBSouth + West Africa & \num{-0.147} \\
& (\num{1.272}) \\
POBIran & \num{3.046}** \\
& (\num{1.031}) \\
POBMiddle East + W

In [19]:
# model 4:
modelsummary(
    linlogRegImmigrantsNOC_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)

\begin{tabular}{ll}
\hline
& (1) \\ \hline
POBCentral America & \num{-0.171} \\
& (\num{0.682}) \\
POBJamaica & \num{-3.338}*** \\
& (\num{0.432}) \\
POBCaribbean & \num{2.685}** \\
& (\num{0.884}) \\
POBSouth America & \num{-0.574} \\
& (\num{0.484}) \\
POBUK & \num{-2.101}* \\
& (\num{0.965}) \\
POBGermany & \num{-1.254} \\
& (\num{1.200}) \\
POBFrance & \num{-4.556} \\
& (\num{3.432}) \\
POBNorthern + Western Europe & \num{-3.270} \\
& (\num{3.294}) \\
POBPoland & \num{-2.210} \\
& (\num{3.348}) \\
POBEastern Europe & \num{-3.263}* \\
& (\num{1.361}) \\
POBItaly & \num{-1.072} \\
& (\num{1.119}) \\
POBPortugal & \num{1.466}* \\
& (\num{0.718}) \\
POBSouthern Europe & \num{-4.059} \\
& (\num{3.389}) \\
POBEastern Africa & \num{3.086}* \\
& (\num{1.301}) \\
POBNorthern Africa & \num{-3.489}* \\
& (\num{1.391}) \\
POBSouth + West Africa & \num{-0.219} \\
& (\num{1.288}) \\
POBIran & \num{-2.628} \\
& (\num{3.177}) \\
POBMiddle East + West Central Asia & \num{-1.296} \\
& (\num{3.135}) 